# 补充竞品航线价格

客流不仅仅取决于航班基本信息和自己的票价，还取决于竞争航班的票价

由于竞品全航线没有经济舱的单独数据，我们暂且用所有票的平均票价（经济舱和商务舱）来代替，效果可能没有使用经济舱票价好，但是这也要是眼下最好的处理方式了

竞争航班是指from和to相同的航线，起飞时间相差在4小时以内的价格最便宜的那趟航班作为竞争航班

如果4小时内没有from和to相同的航班，则以起飞时间最近的那趟航班作为竞争航班

如果历史上都没有from和to相同的航班，那么假定竞争航班票价和该航班票价相同




# 全市场票价计算

In [1]:
import pandas as pd
import numpy as np

# 读取CSV文件

df_all = pd.read_csv('../../data_hh/海航系销售结果数据_2023-01-01_2024-12-31.csv',index_col=0)
df_hh = pd.read_csv('../../data_hh/预处理的数据/pre_2023-2024.csv')


# 显示前两行数据以确保正确加载
print(df_all.shape)
print(df_all.head(5))
print(df_all.tail(5))

# 显示前两行数据以确保正确加载
print(df_hh.shape)
print(df_hh.head(5))
print(df_hh.tail(5))

(8798480, 16)
     flt_date segment flt_no      route    a    b    c bd_type  dep_time  cap  \
0  2023-01-01  TFULXA   111T  TFUGZGLXA  TFU  GZG  LXA      未知  18:00:00    0   
1  2023-01-01  GZGLXA   111T  TFUGZGLXA  TFU  GZG  LXA      未知  20:00:00    0   
2  2023-01-01  TFUGZG   111T  TFUGZGLXA  TFU  GZG  LXA      未知  18:00:00    0   
3  2023-01-01  LXAMIG   3017     LXAMIG  LXA  MIG  NaN      窄体  13:25:00  132   
4  2023-01-01  MIGLXA   3018     MIGLXA  MIG  LXA  NaN      窄体  16:15:00  132   

  aircraft  legs  leg_no  duration   tkt_rev  pax  
0      NaN     3       3      0.00       0.0    0  
1      NaN     3       2      3.00       0.0    0  
2      NaN     3       1      1.00       0.0    0  
3      319     1       1      1.70  223630.0  127  
4      319     1       1      2.28   76854.0   60  
           flt_date segment flt_no      route    a    b    c bd_type  \
8798475  2024-12-20  CGQKWE   8037  CGQTNAKWE  CGQ  TNA  KWE      窄体   
8798476  2024-12-20  CGQNNG   9344  CGQHFEN

## 部分字段统计情况

## 处理pax

In [2]:
# 统计 'pax' 字段中缺失值的行数
missing_pax = df_all[df_all['pax'].isna()]

# 统计 'pax' 字段中0值的行数
zero_pax = df_all[df_all['pax'] == 0]

# 输出统计结果
num_missing = missing_pax.shape[0]
num_zero = zero_pax.shape[0]

print(df_all.shape[0])
print(f"缺失值的行数: {num_missing}")
print(f"为0的行数: {num_zero}")

8798480
缺失值的行数: 0
为0的行数: 217917


In [3]:
# 删除 'pax' 字段为缺失值或为0的行
df_all = df_all.dropna(subset=['pax'])  # 删除pax列中的缺失值行
df_all = df_all[df_all['pax'] != 0]  # 删除pax列中为0的行

# 查看删除后的DataFrame行数
num_rows_after_cleanup = df_all.shape[0]
print(f"删除缺失值或为0的行后，DataFrame一共有 {num_rows_after_cleanup} 行")

删除缺失值或为0的行后，DataFrame一共有 8580563 行


## 处理tht_rev

In [4]:
# 统计 'unit_price' 列中为 0 的行数
zero_count = (df_all['tkt_rev'] == 0).sum()

# 统计 'unit_price' 列中为 NaN 的行数
nan_count = df_all['tkt_rev'].isna().sum()

# 输出结果
print(f"tkt_rev 列中为 0 的行数: {zero_count}")
print(f"tkt_rev 列中为 NaN 的行数: {nan_count}")

tkt_rev 列中为 0 的行数: 1438
tkt_rev 列中为 NaN 的行数: 0


In [5]:
# 过滤出 'tkt_rev' 为 0 的行
zero_tkt_rev = df_all[df_all['tkt_rev'] == 0]

# 统计这些 'tkt_rev' 为 0 的行中 'leg_no' 字段的不同取值及其数量
leg_no_counts = zero_tkt_rev['leg_no'].value_counts()

# 输出统计结果
print(f"tkt_rev 为 0 的行中，'leg_no' 字段的不同取值及其数量：")
print(leg_no_counts)

tkt_rev 为 0 的行中，'leg_no' 字段的不同取值及其数量：
leg_no
3    1121
1     225
2      91
6       1
Name: count, dtype: int64


In [6]:
df_all = df_all[df_all['tkt_rev'] != 0]

# 计算单价 'unit_price'，即 tkt_rev 除以 pax
df_all['unit_price'] = df_all['tkt_rev'] / df_all['pax']

# 删除 'tkt_rev' 列
df_all = df_all.drop(columns=['tkt_rev'])

# 查看结果
print(df_all.shape)
print(df_all.head(2))
print(df_all.tail(2))

(8579125, 16)
     flt_date segment flt_no   route    a    b    c bd_type  dep_time  cap  \
3  2023-01-01  LXAMIG   3017  LXAMIG  LXA  MIG  NaN      窄体  13:25:00  132   
4  2023-01-01  MIGLXA   3018  MIGLXA  MIG  LXA  NaN      窄体  16:15:00  132   

  aircraft  legs  leg_no  duration  pax   unit_price  
3      319     1       1      1.70  127  1760.866142  
4      319     1       1      2.28   60  1280.900000  
           flt_date segment flt_no   route    a    b    c bd_type  dep_time  \
8798478  2024-12-25  SHAPKX   6874  SHAPKX  SHA  PKX  NaN      窄体  09:40:00   
8798479  2024-12-29  TFUPVG   5296  TFUPVG  TFU  PVG  NaN      窄体  07:15:00   

         cap aircraft  legs  leg_no  duration  pax  unit_price  
8798478  175      321     1       1      1.85  152  624.013158  
8798479  158      320     1       1      2.02  139  412.719424  


# 为海航数据补充竞争航线价格

In [7]:
import pandas as pd
import numpy as np
from datetime import timedelta
from tqdm import tqdm  # 导入 tqdm

# 假设 df_all 和 df_hh 已经加载
# df_all = pd.read_csv('路径')  # 已加载的数据
# df_hh = pd.read_csv('路径')  # 已加载的数据

# 确保 df_all 和 df_hh 中 'dep_time' 和 'flt_date' 是正确的 datetime 类型
df_all['dep_time'] = pd.to_datetime(df_all['flt_date'].astype(str) + ' ' + df_all['dep_time'].astype(str), format='%Y-%m-%d %H:%M:%S')
df_hh['dep_time'] = pd.to_datetime(df_hh['flt_date'].astype(str) + ' ' + df_hh['dep_time'].astype(str), format='%Y-%m-%d %H:%M:%S')

# 1. 按 'segment' 字段分组
df_all_grouped = df_all.groupby('segment')

# 2. 准备一个空的列表用于存储竞争航班的票价
competitor_prices = []

# 3. 使用 tqdm 包装 df_hh.iterrows() 来显示进度条
for idx, row_hh in tqdm(df_hh.iterrows(), total=df_hh.shape[0], desc="Processing df_hh rows"):
    segment_hh, dep_time_hh = row_hh['segment'], row_hh['dep_time']
    
    # 获取与当前航班相同的 segment 的所有航班
    df_competing = df_all_grouped.get_group(segment_hh).copy() if segment_hh in df_all_grouped.groups else pd.DataFrame()

    if df_competing.empty:
        # 如果没有相同的航班，直接返回当前航班的票价
        competitor_prices.append(row_hh['unit_price'])
        continue
    
    # 4. 计算时间差
    time_diff = abs(df_competing['dep_time'] - dep_time_hh)
    
    # 5. 筛选出起飞时间与 df_hh 当前航班相差不超过 4 小时的航班
    df_competing.loc[:, 'time_diff'] = time_diff
    df_competing_4h = df_competing[df_competing['time_diff'].abs() <= timedelta(hours=4)]  # 使用 abs() 来计算绝对时间差
    
    if not df_competing_4h.empty:
        # 6. 如果有符合条件的航班，选择最便宜的
        cheapest_competing = df_competing_4h.loc[df_competing_4h['unit_price'].idxmin()]
        competitor_prices.append(cheapest_competing['unit_price'])
    else:
        # 7. 如果没有符合条件的航班，选择最接近的航班（按时间差最小）
        closest_competing = df_competing.loc[df_competing['time_diff'].idxmin()]
        competitor_prices.append(closest_competing['unit_price'])

# 将竞争航班的票价添加到 df_hh
df_hh['competitor_price'] = competitor_prices

# 查看结果
print(df_hh[['segment', 'dep_time', 'unit_price', 'competitor_price']].head())

Processing df_hh rows: 100%|███████| 1611142/1611142 [3:02:49<00:00, 146.88it/s]


  segment            dep_time   unit_price  competitor_price
0  AATURC 2023-01-01 14:35:00   470.474227        539.272727
1  ACFURC 2023-01-01 22:40:00   454.925373        578.797468
2  ACFXIY 2023-01-01 18:00:00  1177.818182       1177.818182
3  AKAHGH 2023-01-01 12:55:00   669.090909        669.090909
4  AKUCGO 2023-01-01 13:10:00  1794.783133       1859.812121


In [8]:
df_hh.to_csv('data_with_competitor_prices.csv')

In [9]:
df_hh.head(5)

,flt_date,segment,flt_no,dep_time,cap,aircraft,legs,leg_no,duration,pax,a,b,c,unit_price,competitor_price
0,2023-01-01,AATURC,7558,2023-01-01 14:35:00,110,195,1,1,1.30,97,AAT,URC,NaN,470.474227,539.272727
1,2023-01-01,ACFURC,7470,2023-01-01 22:40:00,110,195,1,1,1.25,67,ACF,URC,NaN,454.925373,578.797468
2,2023-01-01,ACFXIY,769R,2023-01-01 18:00:00,0,190,3,3,4.75,22,ACF,TLQ,XIY,1177.818182,1177.818182
3,2023-01-01,AKAHGH,5248,2023-01-01 12:55:00,162,320,1,1,1.83,55,AKA,HGH,NaN,669.090909,669.090909
4,2023-01-01,AKUCGO,6250,2023-01-01 13:10:00,167,320,1,1,3.98,166,AKU,CGO,NaN,1794.783133,1859.812121
